Accelerator: GPU T4 x2, Internet on, and attach the `kaggle_prepare_data` output under Add Input → Your Work → Notebook Output.

In [ ]:
REPO_URL = "https://github.com/Splestule/candidate_reranker.git"
BRANCH = "main"

In [ ]:
import subprocess, sys
from pathlib import Path

CODE = Path("/kaggle/working/candidate_reranker")
if not (CODE / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(CODE)],
                   check=True)

sys.path.insert(0, str(CODE / "src"))
import kaggle_env as K

COMMIT = K.sync(REPO_URL, BRANCH)     # rerun this cell after every push
K.gpu_info()
env = K.prepare(COMMIT)

In [ ]:
assert K.run(env, "selftest.py",
             "--work", "/kaggle/working",
             "--n_utts", "3",
             "--n_candidates", "5") == 0

Manifest for the dialect audio. Paths differ between notebooks, so it is built here.

In [ ]:
DIALECTS = env.results / "dialects.jsonl"
K.run(env, "make_manifest.py",
      "--slr83", *sorted(p for p in env.ood.iterdir() if p.is_dir()),
      "--out", DIALECTS)

Dump candidates. Roughly 2 hours of GPU for all four.

In [ ]:
def dump(tag, source, path, **kw):
    args = ["--source", source, "--path", path,
            "--base_model", env.base_model, "--adapter", env.adapter,
            "--out", env.results / f"{tag}-{COMMIT}.jsonl",
            "--tag", tag, "--resume"]
    for k, v in kw.items():
        args += [f"--{k}", v]
    return K.run(env, "dump_candidates.py", *args)

dump("test-clean", "librispeech", env.librispeech / "test-clean")
dump("test-other", "librispeech", env.librispeech / "test-other")
dump("dialects", "manifest", DIALECTS)
dump("dev-clean", "librispeech", env.librispeech / "dev-clean", max_utts=800)

Tune on dev, then measure on all three test sets.

In [ ]:
K.run(env, "tune.py",
      "--dev", env.results / f"dev-clean-{COMMIT}.jsonl",
      "--test", env.results / f"test-clean-{COMMIT}.jsonl",
      "--json", env.results / f"tune-{COMMIT}.json")

In [ ]:
for tag in ["test-clean", "test-other", "dialects"]:
    print("\n" + "#" * 74 + f"\n# {tag}\n" + "#" * 74)
    K.run(env, "analyze.py", env.results / f"{tag}-{COMMIT}.jsonl",
          "--json", env.results / f"{tag}-{COMMIT}.stats.json")
    K.run(env, "analyze_compose.py", env.results / f"{tag}-{COMMIT}.jsonl",
          "--json", env.results / f"compose-{tag}-{COMMIT}.json")